In [1]:
import pandas as pd
from pathlib import Path
import sys
sys.path.append("../src")
from preprocessing import create_preprocessor

FE_DF_PATH = Path("../data/processed/paysim_feature_engineered.parquet")
fe_df = pd.read_parquet(FE_DF_PATH)

In [2]:
# look at the fe df
fe_df.head()

,step,type,amount,oldBalanceOrig,newBalanceOrig,oldBalanceDest,newBalanceDest,isFlaggedFraud,senderBalanceError,receiverBalanceError,netChangeOrig,netChangeDest,isLargeTransaction,day,hour,dayOfWeek,amountToBalanceRatio,isFraud
0,1,PAYMENT,9839.639648,170136.0,160296.359375,0.0,0.0,0,0,1,-9839.640625,0.0,0,0,1,0,0.057834,0
1,1,PAYMENT,1864.280029,21249.0,19384.720703,0.0,0.0,0,0,1,-1864.279297,0.0,0,0,1,0,0.087731,0
2,1,TRANSFER,181.000000,181.0,0.000000,0.0,0.0,0,0,1,-181.000000,0.0,0,0,1,0,0.994505,1
3,1,CASH OUT,181.000000,181.0,0.000000,21182.0,0.0,0,0,1,-181.000000,-21182.0,0,0,1,0,0.994505,1
4,1,PAYMENT,11668.139648,41554.0,29885.859375,0.0,0.0,0,0,1,-11668.140625,0.0,0,0,1,0,0.280788,0


In [3]:
# create train and test sets
from sklearn.model_selection import train_test_split

X = fe_df.drop("isFraud", axis=1)
y = fe_df.isFraud

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [4]:
# choose num. and cat. features
num_features = X.select_dtypes(include="number").columns
cat_features = X.select_dtypes(exclude="number").columns

In [5]:
# create pipelines
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.dummy import DummyClassifier

preprocessor = create_preprocessor(num_features, cat_features)

# handle XG Boost imbalance
fraud_count = y_train.sum()
normal_count = len(y_train) - fraud_count
scale_pos_weight = normal_count / fraud_count

pipelines = {
    "Dummy": Pipeline([
        ("preprocessor", preprocessor),
        ("model", DummyClassifier(strategy="most_frequent", random_state=42))
    ]),
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(random_state=42, class_weight="balanced", max_iter=1000)),
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced", n_jobs=-1)),
    ]),
    "XG Boost": Pipeline([
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1, scale_pos_weight=scale_pos_weight)),
    ]),
}

In [6]:
# create metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score
from sklearn.model_selection import cross_val_score
import joblib

results = {}

for model_name, pipeline in pipelines.items():
    cv_scores = cross_val_score(pipeline, 
                                X_train, 
                                y_train,
                                cv=3,
                                scoring="f1")
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    results[model_name] = {
        "CV Mean F1": cv_scores.mean(),
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred),
        "ROC AUC Score": roc_auc_score(y_test, y_proba),
        "F1 Score": f1_score(y_test, y_pred)
    }
    filename = model_name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(pipeline, f"../outputs/models/{filename}")

In [7]:
# save the metrics dataframe
metrics_df = pd.DataFrame(results).T
metrics_df.to_csv("../data/processed/model_metrics.csv", index=True)